# Model 2: MuRIL — Multilingual Representations for Indian Languages
## Cross-Lingual Transfer: Hinglish → Hindi / English / Combined

**Model:** `google/muril-base-cased`  
**Why MuRIL?** Google's MuRIL is trained on 17 Indian languages + their transliterated forms.
It explicitly includes Hinglish-style transliterated data in pretraining.
Expected to outperform XLM-RoBERTa on Hindi transfer and match it on English.

**Key comparison:** MuRIL vs XLM-RoBERTa on the same 3 test sets.


## Step 0: Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers==4.40.0 datasets==2.19.0 scikit-learn lime shap accelerate seaborn
print("Done")

## Step 1: Setup

In [ ]:
import os,re,json,random,warnings
import numpy as np,pandas as pd
import matplotlib.pyplot as plt,seaborn as sns
import torch
from torch import nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix,f1_score
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,
                           TrainingArguments,Trainer,EarlyStoppingCallback,set_seed)
from datasets import Dataset as HFDataset,DatasetDict
warnings.filterwarnings('ignore')
SEED=42;set_seed(SEED);random.seed(SEED);np.random.seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_LEN=128;BATCH_SIZE=16;EPOCHS=5;LR=2e-5;NUM_LABELS=2
LABEL_MAP={0:"Non-Hate",1:"Hate"}
OUTPUT_DIR="/content/drive/MyDrive/muril_results"
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
MODEL_NAME="google/muril-base-cased"
print(f"Device:{DEVICE} | Model:{MODEL_NAME}")

## Step 2: Normalization Module (same as Model 1)

In [ ]:
SLANG={
    r'\bbc\b':'bhenchod',r'\bmc\b':'madarchod',r'\bbkl\b':'bhosadike',
    r'\bbsdk\b':'bhosadike',r'\bch[ou]tiy[ae]?\b':'chutiya',r'\bchutia\b':'chutiya',
    r'\bsaale?\b':'saala',r'\bkamine[y]?\b':'kamina',r'\bkutt[ey]?\b':'kutta',
    r'\bgand[uu]?\b':'gandu',r'\bharami[i]?\b':'harami',r'\bbhench[o0]d\b':'bhenchod',
    r'\brandi\b':'randi',r'\bnalayak\b':'nalayak',r'\bghati[y]?a\b':'ghatiya',
    r'\bneech\b':'neech',r'\bgawar\b':'gawar',r'\bnafrat\b':'nafrat',
    r'\bnhi\b':'nahi',r'\bnai\b':'nahi',r'\bkl\b':'kal',r'\bkyu\b':'kyun',
    r'\blol\b':'laughing',r'\bwtf\b':'what the hell',
}
TRANSLIT={
    r'\bkyaa\b':'kya',r'\bnahin\b':'nahi',r'\bnaheen\b':'nahi',
    r'\bbhaut\b':'bahut',r'\bbohat\b':'bahut',r'\bbohot\b':'bahut',
    r'\bterii\b':'teri',r'\bmujhey\b':'mujhe',r'\byar\b':'yaar',
    r'\bachha\b':'accha',r'\bacha\b':'accha',
}
class HinglishNormalizer:
    def normalize(self,text):
        t=text.lower().strip()
        t=re.sub(r'http\S+|www\.\S+','',t)
        t=re.sub(r'@\w+','[USER]',t);t=re.sub(r'#(\w+)',r'\1',t)
        t=re.sub(r'([!?.])\1{2,}',r'\1',t);t=re.sub(r'^rt\s+','',t)
        t=re.sub(r'(.)\1{2,}',r'\1\1',t)
        for p,v in TRANSLIT.items(): t=re.sub(p,v,t,flags=re.IGNORECASE)
        for p,v in SLANG.items(): t=re.sub(p,v,t,flags=re.IGNORECASE)
        return re.sub(r'\s+',' ',t).strip()
    def normalize_batch(self,texts): return [self.normalize(t) for t in texts]
    def augment(self,texts,labels):
        n=self.normalize_batch(texts)
        return list(texts)+n,list(labels)+list(labels)
normalizer=HinglishNormalizer()
print("Normalizer ready")

## Step 3: Load Data (same demo or upload CSVs)

In [ ]:
def make_demo():
    h_hate=["teri maa ki aankh sale harami","bc mar ja tu yahan se",
             "yeh sab chutiye hain desh ke","kamine log hain yeh sab",
             "bhenchod nikal yahan se","harami kutta hai yeh banda",
             "yeh muslims sab terrorist hain","bsdk kya bakwas kar raha hai",
             "teri behen randi hai","bhikhari kahin ke","neech zaat ke log",
             "janwar hain yeh log","gawar anpadh hain yeh","desh ke gaddar hain",
             "pagal kutte ko shoot karo","kutton ki tarah bhonkte hain",
             "in logo ko desh se nikalo","yeh log ganda khoon hain",
             "bkl apna kaam kar","gandu hai tu bilkul"]
    h_nhate=["aaj ka din bahut accha raha","mujhe yeh movie pasand aayi",
              "good morning everyone","yeh khana tasty hai yaar",
              "kal college jaana hai","bhai kaise ho tum sab",
              "cricket match mein India jeet gayi","naya phone kharida",
              "programming seekhna chahta hun","yeh weather pleasant hai",
              "music sunna pasand hai","congratulations yaar result aayi",
              "bahut helpful raha article","positive soch rakho hamesha",
              "grateful hun sab ke liye","padhai mein mann laga hun",
              "aaj gym gaya pehli baar","chai peelo yaar subah subah",
              "family ke saath enjoy kar raha hun","yeh novel padh raha hun"]
    hi_hate=["यह लोग देश के दुश्मन हैं","इन्हें देश से निकालो",
              "नालायक लोग हैं यह सब","घटिया इंसान है यह",
              "बेकार है यह सब कुछ","देश के गद्दार हैं यह",
              "यह समुदाय अपराधी है","नीच जाति के लोग",
              "जानवर हैं यह लोग","नफरत है मुझे इनसे"]
    hi_nhate=["आज का दिन बहुत अच्छा था","परिवार के साथ समय बिताया",
               "बहुत अच्छा खाना बनाया","दोस्त से मुलाकात हुई",
               "नया काम शुरू किया","बहुत खुशी हुई सुनकर",
               "आज बहुत अच्छा मौसम है","परीक्षा में अच्छे अंक आए",
               "अच्छी फिल्म देखी आज","खुश हूं आज बहुत"]
    en_hate=["These people should be deported","I hate this community they are criminals",
              "Women belong in the kitchen","This religion should be banned",
              "They are taking over our country","These immigrants are ruining everything",
              "All of them are terrorists","They should go back to their country",
              "This group is a plague","These people are subhuman animals"]
    en_nhate=["Had a wonderful day at the park","Just finished reading a great book",
               "The weather is so beautiful","Learning Python has been fun",
               "Family dinner was amazing","Good morning hope you have great day",
               "Just got a new phone loving it","Finished my project ahead of schedule",
               "Looking forward to the weekend","Grateful for all the good things"]
    df_tr=pd.DataFrame({'text':h_hate*4+h_nhate*4,'label':[1]*80+[0]*80}).sample(frac=1,random_state=SEED).reset_index(drop=True)
    df_hi=pd.DataFrame({'text':hi_hate+hi_nhate,'label':[1]*10+[0]*10})
    df_en=pd.DataFrame({'text':en_hate+en_nhate,'label':[1]*10+[0]*10})
    df_co=pd.concat([df_tr.sample(15,random_state=SEED),df_hi.sample(10,random_state=SEED),df_en.sample(10,random_state=SEED)]).sample(frac=1,random_state=SEED).reset_index(drop=True)
    return df_tr,df_hi,df_en,df_co
df_train_raw,df_hindi,df_english,df_combined=make_demo()
print(f"Train:{len(df_train_raw)} | Hindi:{len(df_hindi)} | English:{len(df_english)} | Combined:{len(df_combined)}")

## Step 4: Augment + Split

In [ ]:
aug_t,aug_l=normalizer.augment(df_train_raw['text'].tolist(),df_train_raw['label'].tolist())
df_aug=pd.DataFrame({'text':aug_t,'label':aug_l}).sample(frac=1,random_state=SEED).reset_index(drop=True)
df_tr,df_dev=train_test_split(df_aug,test_size=0.15,stratify=df_aug['label'],random_state=SEED)
print(f"Train:{len(df_tr)} | Dev:{len(df_dev)}")

## Step 5: Train MuRIL

> MuRIL is pretrained on 17 Indian languages including transliterated Hindi.
> It understands Hinglish code-mixing better than generic XLM-RoBERTa.

In [ ]:
def get_cw(df):
    c=df['label'].value_counts().sort_index().values.astype(float)
    w=1.0/c;w=w/w.sum()*len(c);return torch.tensor(w,dtype=torch.float)
def tok_ds(tr,dv,te,tok):
    def fn(b): return tok(b['text'],truncation=True,padding='max_length',max_length=MAX_LEN)
    ds=DatasetDict({'train':HFDataset.from_pandas(tr[['text','label']]),
                    'validation':HFDataset.from_pandas(dv[['text','label']]),
                    'test':HFDataset.from_pandas(te[['text','label']])})
    ds=ds.map(fn,batched=True,remove_columns=['text'])
    ds=ds.rename_column('label','labels');ds.set_format('torch');return ds
class WTrainer(Trainer):
    def __init__(self,cw,*a,**k): super().__init__(*a,**k);self.cw=cw.to(DEVICE)
    def compute_loss(self,m,inp,ro=False,**k):
        lb=inp.pop('labels');o=m(**inp)
        loss=nn.CrossEntropyLoss(weight=self.cw)(o.logits,lb)
        return (loss,o) if ro else loss
def cm_fn(ep):
    p=np.argmax(ep.predictions,axis=-1)
    return {'macro_f1':f1_score(ep.label_ids,p,average='macro')}

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=NUM_LABELS)
ds=tok_ds(df_tr,df_dev,df_dev,tokenizer)
args=TrainingArguments(output_dir=f"{OUTPUT_DIR}/ckpt",num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,warmup_steps=200,weight_decay=0.01,
    eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,
    metric_for_best_model='macro_f1',greater_is_better=True,
    fp16=torch.cuda.is_available(),seed=SEED,report_to='none',save_total_limit=1)
trainer=WTrainer(get_cw(df_tr),model=model,args=args,
                  train_dataset=ds['train'],eval_dataset=ds['validation'],
                  compute_metrics=cm_fn,callbacks=[EarlyStoppingCallback(2)])
trainer.train()
trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
print("MuRIL training complete")

## Step 6: Evaluate on 3 Test Sets

In [ ]:
def evaluate_on(model,tokenizer,df_test,name):
    model.eval();preds=[]
    for text in df_test['text'].tolist():
        enc=tokenizer(str(text),return_tensors='pt',truncation=True,padding=True,max_length=MAX_LEN).to(DEVICE)
        with torch.no_grad(): logits=model(**enc).logits
        preds.append(torch.argmax(logits).item())
    true=df_test['label'].tolist()
    rpt=classification_report(true,preds,target_names=list(LABEL_MAP.values()),output_dict=True)
    print(f"\n[{name}]"); print(classification_report(true,preds,target_names=list(LABEL_MAP.values())))
    return {'dataset':name,'macro_f1':round(rpt['macro avg']['f1-score'],4),
            'weighted_f1':round(rpt['weighted avg']['f1-score'],4),
            'precision':round(rpt['macro avg']['precision'],4),
            'recall':round(rpt['macro avg']['recall'],4),
            'preds':np.array(preds),'true':np.array(true)}

r_h=evaluate_on(model,tokenizer,df_hindi,"Hindi")
r_e=evaluate_on(model,tokenizer,df_english,"English")
r_c=evaluate_on(model,tokenizer,df_combined,"Combined")
all_results=[r_h,r_e,r_c]
rdf=pd.DataFrame([{k:v for k,v in r.items() if k not in ['preds','true']} for r in all_results])
print("\nMuRIL Results:"); print(rdf[['dataset','macro_f1','weighted_f1']].to_string(index=False))
rdf.to_csv(f"{OUTPUT_DIR}/muril_results.csv",index=False)

## Step 7: XAI Analysis

In [ ]:
def pred_fn(texts):
    model.eval();all_p=[]
    for i in range(0,len(texts),8):
        enc=tokenizer(list(texts[i:i+8]),return_tensors='pt',truncation=True,padding=True,max_length=MAX_LEN).to(DEVICE)
        with torch.no_grad(): logits=model(**enc).logits
        all_p.extend(torch.softmax(logits,-1).cpu().numpy())
    return np.array(all_p)

def aopc(texts,n=5):
    drops=[]
    for t in texts:
        toks=t.split()
        if len(toks)<3: continue
        orig=pred_fn([t])[0].max();step=max(1,len(toks)//n);cum=0.0;rem=toks.copy()
        for _ in range(n):
            if not rem: break
            rem=rem[:-step]
            cum+=float(orig-pred_fn([' '.join(rem) if rem else '[MASK]'])[0].max())
        drops.append(cum/n)
    return float(sum(drops)/len(drops)) if drops else 0.0

xai_rows=[]
for df_t,name in [(df_hindi,'Hindi'),(df_english,'English'),(df_combined,'Combined')]:
    txts=df_t['text'].head(15).tolist()
    ntxts=normalizer.normalize_batch(txts)
    ar,an=aopc(txts),aopc(ntxts)
    xai_rows.append({'dataset':name,'aopc_raw':round(ar,4),'aopc_norm':round(an,4),'delta':round(an-ar,4)})
    print(f"[{name}] AOPC Raw:{ar:.4f} Norm:{an:.4f} Δ={an-ar:+.4f}")

pd.DataFrame(xai_rows).to_csv(f"{OUTPUT_DIR}/muril_xai.csv",index=False)

fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,row in zip(axes,xai_rows):
    ar,an=row['aopc_raw'],row['aopc_norm']
    bars=ax.bar(['Raw','Norm'],[ar,an],color=['#B5D4F4','#185FA5'],width=0.4)
    for bar,v in zip(bars,[ar,an]):
        ax.text(bar.get_x()+bar.get_width()/2.,bar.get_height()+.001,f'{v:.4f}',ha='center',fontweight='bold')
    ax.set_title(f"AOPC — {row['dataset']}");ax.set_ylabel('AOPC');ax.grid(axis='y',alpha=0.3)
    ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
plt.suptitle('MuRIL — XAI Faithfulness Across 3 Test Sets',fontsize=13)
plt.tight_layout();plt.savefig(f"{OUTPUT_DIR}/muril_aopc.png",dpi=150);plt.show()

# Confusion matrices
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,r in zip(axes,all_results):
    cm=confusion_matrix(r['true'],r['preds'])
    sns.heatmap(cm,annot=True,fmt='d',cmap='Greens',ax=ax,
                xticklabels=list(LABEL_MAP.values()),yticklabels=list(LABEL_MAP.values()))
    ax.set_title(r['dataset'])
plt.suptitle('MuRIL Confusion Matrices',fontsize=13)
plt.tight_layout();plt.savefig(f"{OUTPUT_DIR}/muril_cm.png",dpi=150);plt.show()

In [ ]:
print("="*60,"  MuRIL COMPLETE RESULTS","="*60,sep="\n")
print(rdf[['dataset','macro_f1','weighted_f1','precision','recall']].to_string(index=False))
print("\nXAI Faithfulness:")
print(pd.DataFrame(xai_rows).to_string(index=False))